# Hands-on with Kafka

![kafka](https://images.contentful.com/gt6dp23g0g38/53UO4964r0e7kRVm0mcUUZ/f6f6d7b1b90e8e88a5be0d1845bdf950/what_is_kafka_and_how_does_it_work.png)

## Installing Kafka Python Client

First, we'll need to install Kafka Python client `kafka-python` to consume messages from a Kafka cluster.

In [1]:
import json
import ssl
from kafka import KafkaConsumer
from pprint import pprint
from dotenv import load_dotenv
import os

load_dotenv()


HOST = os.getenv("BOOTSTRAP_SERVERS")
PORT = os.getenv("PORT")
BOOTSTRAP_SERVERS = f"{HOST}:{PORT}"
AIVEN_USERNAME = os.getenv("AIVEN_USERNAME")
AIVEN_PASSWORD = os.getenv("AIVEN_PASSWORD")
CONSUMER_GROUP = 'pizza-consumer-group'
TOPIC_NAME = os.getenv("TOPIC_NAME")

First, we'll instantiate the consumer.

Our data was serialized in JSON during publishing, hence we'll need to deserialize as JSON when consuming. In actual production, it is recommended to use the binary format that we've learnt, such as Avro.

In [2]:
ssl_context = ssl.create_default_context(
    cafile="../ca.pem"
)

consumer = KafkaConsumer(
    TOPIC_NAME,
    bootstrap_servers=BOOTSTRAP_SERVERS,
    security_protocol='SASL_SSL',
    sasl_mechanism='SCRAM-SHA-256',
    sasl_plain_username=AIVEN_USERNAME,
    sasl_plain_password=AIVEN_PASSWORD,
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    value_deserializer=lambda v: json.loads(v.decode('utf-8')),
    key_deserializer=lambda v: json.loads(v.decode('utf-8')) if v is not None else None,
    ssl_context=ssl_context,
    group_id=CONSUMER_GROUP,
)
print(f"Consumer created and subscribed to topic: {TOPIC_NAME}")

Consumer created and subscribed to topic: pizza-orders


We set the offset for the consumer to be `earliest` to get the earliest order (message).

List the available topics:

In [3]:
# List available topics using the Kafka client
print("Available topics:")
print(consumer.topics())

Available topics:
{'pizza-orders'}


Let's assume we are the owners of a pizza delivery chain, and we'd like to push the orders to Apache Kafka as they come in.

We have a topic consisting of pizza orders, which contains the Order ID, Order Details (pizzas and toppings), client's Name, Address, Phone Number, the Shop Name, Total Cost and Timestamp.

There is only one partition for the topic. First, let's subscribe to the topic:

In [4]:
topic_name = TOPIC_NAME

In [5]:
# Already subscribed to the topic during consumer creation
print(f"Subscribed to topic: {topic_name}")

Subscribed to topic: pizza-orders


We can then start reading the events, let's read and (pretty) print the first 5:

In [6]:
# Read and pretty-print the first 5 messages
from pprint import pprint

In [7]:
messages_count = 0
for message in consumer:
    print(f"Topic: {message.topic}, Partition: {message.partition}, Offset: {message.offset}")
    print(f"Key: {message.key}")
    pprint("-------------------")
    pprint(message.value)
    print()
    messages_count += 1
    if messages_count >= 5:
        break

Topic: pizza-orders, Partition: 0, Offset: 0
Key: {'shop': 'Circular Pi Pizzeria'}
'-------------------'
{'address': '21087 Calvin Plains\nJonesland, NY 76392',
 'cost': 40.37,
 'id': 0,
 'name': 'Jessica Smith',
 'phoneNumber': '001-701-915-3000',
 'pizzas': [{'additionalToppings': [], 'pizzaName': 'Salami'},
            {'additionalToppings': ['🐟 tuna'], 'pizzaName': 'Peperoni'},
            {'additionalToppings': ['🫑 green peppers', '🍅 tomato', '🧄 garlic'],
             'pizzaName': 'Marinara'}],
 'shop': 'Circular Pi Pizzeria',
 'timestamp': 1780996813910}

Topic: pizza-orders, Partition: 0, Offset: 1
Key: {'shop': 'Ill Make You a Pizza You Cant Refuse'}
'-------------------'
{'address': '19721 Drew Key\nNew Donaldport, NH 05690',
 'cost': 35.23,
 'id': 1,
 'name': 'Roger Brown',
 'phoneNumber': '475-943-3780x8105',
 'pizzas': [{'additionalToppings': ['🍌 banana',
                                    '🌶️ hot pepper',
                                    '🧀 blue cheese'],
             

Now, continue printing the next 5 events, notice the offset number and order id:

In [11]:
messages_count = 0
for message in consumer:
    print(f"Topic: {message.topic}, Partition: {message.partition}, Offset: {message.offset}")
    pprint(message.value)
    print()
    messages_count += 1
    if messages_count >= 5:
        break

Topic: pizza-orders, Partition: 0, Offset: 5
{'address': '144 Ross Port\nAliciafurt, KY 28535',
 'cost': 29.16,
 'id': 0,
 'name': 'Ricky Sexton',
 'phoneNumber': '8667536484',
 'pizzas': [{'additionalToppings': ['🍅 tomato', '🧄 garlic'],
             'pizzaName': 'Marinara'}],
 'shop': 'Circular Pi Pizzeria',
 'timestamp': 1780996873936}

Topic: pizza-orders, Partition: 0, Offset: 6
{'address': '49595 Gary Port Suite 707\nNorth Michaelfort, CA 18853',
 'cost': 11.87,
 'id': 1,
 'name': 'Russell Sanders',
 'phoneNumber': '(873)972-6401x764',
 'pizzas': [{'additionalToppings': ['🥓 bacon'], 'pizzaName': 'Mari & Monti'}],
 'shop': 'Ill Make You a Pizza You Cant Refuse',
 'timestamp': 1780996875939}

Topic: pizza-orders, Partition: 0, Offset: 7
{'address': '4189 Madison Glens Suite 206\nLake Vickiborough, TN 20792',
 'cost': 46.54,
 'id': 2,
 'name': 'Stephanie Rivera',
 'phoneNumber': '(544)795-9493x841',
 'pizzas': [{'additionalToppings': ['🧄 garlic', '🐟 tuna', '🍅 tomato'],
             '

In [16]:
# To reset to the beginning, create a new consumer group or use a different group ID
print("Note: Kafka consumers maintain offsets in the broker.")
print("To restart from beginning, use a new group ID or reset consumer offsets.")

Note: Kafka consumers maintain offsets in the broker.
To restart from beginning, use a new group ID or reset consumer offsets.


In [9]:
# Reset consumer offsets to the beginning of the topic
consumer.seek_to_beginning()
print("Consumer offsets reset to the beginning of the topic.")

Consumer offsets reset to the beginning of the topic.


In [12]:
consumer.close()
print("Consumer closed successfully")

Consumer closed successfully
